# Fetch and Clean: Indeed AI Job Posting Share

This notebook cleans the raw Indeed AI hiring share dataset 
sourced from Indeed's Hiring Lab. The dataset tracks the daily 
share of job postings that mention AI as a required skill, 
covering 9 countries from 2019 to 2026. The output file 
`indeedAI_clean.csv` is used in `indeed_analysis_final.ipynb`.

## Dataset Overview

| | Raw | After Cleaning |
|---|---|---|
| **Rows** | ~23,000 | ~23,000 |
| **Columns** | 6 | 4 |
| **Source** | `ai-headline-share.csv` | `indeedAI_clean.csv` |

## Cleaning Steps

1. **Drop redundant columns:** `__typename` (constant metadata) 
   and `aiType` (single-value field, no analytical value)
2. **Date conversion:** `dateString` (str) to `date` (datetime), 
   invalid dates coerced to NaT, original column dropped
3. **Column renaming:** standardised to snake_case
4. **Sorting:** by `date` then `country`

## Final Schema

| Column | Type | Description |
|---|---|---|
| `date` | datetime | Date of observation |
| `country_code` | str | ISO country code |
| `country` | str | Full country name |
| `ai_share_pct` | float | Share (%) of job postings mentioning AI |

# 1. Importing the Dataset

In [ ]:
import pandas as pd
import numpy as np
df_raw = pd.read_csv("IndeedAI_raw.csv" )

# 2. Exploring the Dataset

In [ ]:
print(df_raw.shape)
df_raw.head()

(23292, 6)


,__typename,dateString,countryCode,countryName,aiType,value
0,HiringLabNationalAI,2019-01-01,AU,Australia,NaN,2.31
1,HiringLabNationalAI,2019-01-01,CA,Canada,NaN,1.80
2,HiringLabNationalAI,2019-01-01,DE,Germany,NaN,2.00
3,HiringLabNationalAI,2019-01-01,FR,France,NaN,2.58
4,HiringLabNationalAI,2019-01-01,GB,United Kingdom,NaN,2.23


In [ ]:
# 23k rows, 6 columns in dataset

In [ ]:
df_raw.columns

Index(['__typename', 'dateString', 'countryCode', 'countryName', 'aiType',
       'value'],
      dtype='object')

In [ ]:
df_raw.dtypes

__typename      object
dateString      object
countryCode     object
countryName     object
aiType         float64
value          float64
dtype: object

In [ ]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23292 entries, 0 to 23291
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   __typename   23292 non-null  object 
 1   dateString   23292 non-null  object 
 2   countryCode  23292 non-null  object 
 3   countryName  23292 non-null  object 
 4   aiType       0 non-null      float64
 5   value        23292 non-null  float64
dtypes: float64(2), object(4)
memory usage: 1.1+ MB


In [ ]:
#Date range:
print("Date min: ",df_raw["dateString"].min(), "\nDate max: ", df_raw["dateString"].max())

Date min:  2019-01-01 
Date max:  2026-01-31


In [ ]:
# No of countries:
print("Countries:", df_raw["countryCode"].nunique(), df_raw["countryName"].nunique())

Countries: 9 9


In [ ]:
df_raw.describe()

,aiType,value
count,0.0,23292.000000
mean,NaN,2.897433
std,NaN,1.373625
min,NaN,1.460000
25%,NaN,2.070000
50%,NaN,2.440000
75%,NaN,3.230000
max,NaN,12.610000


In [ ]:
df_raw.__typename.nunique()
# check __typename has only one unique value (constant metadata, safe to drop)


1

# 3. Cleaning the Dataset

In [ ]:
# dropping redundant columns
df= df_raw.drop(["__typename","aiType"],axis=1)


In [ ]:
#converting date to datetime dtype
df["date"] = pd.to_datetime(df_raw["dateString"], errors="coerce")
df.drop("dateString",axis=1,inplace=True)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23292 entries, 0 to 23291
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   countryCode  23292 non-null  object        
 1   countryName  23292 non-null  object        
 2   value        23292 non-null  float64       
 3   date         23292 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(1), object(2)
memory usage: 728.0+ KB


In [ ]:
# renaming columns
df = df.rename(columns={
    "countryCode": "country_code",
    "countryName": "country",
    "value": "ai_share_pct"
})

In [ ]:
# sorting dataset based on ascending order of date and country
df = df.sort_values(["date","country"])
df

,country_code,country,ai_share_pct,date
0,AU,Australia,2.31,2019-01-01
1,CA,Canada,1.80,2019-01-01
3,FR,France,2.58,2019-01-01
2,DE,Germany,2.00,2019-01-01
5,IE,Ireland,4.21,2019-01-01
...,...,...,...,...
23288,IE,Ireland,11.97,2026-01-31
23289,IT,Italy,3.72,2026-01-31
23290,NL,Netherlands,2.63,2026-01-31
23287,GB,United Kingdom,7.02,2026-01-31


In [ ]:
# saving cleaned daatset to a csv
df.to_csv("indeedAI_clean.csv", index=False)